# Stage 5: Data Splitting & Formatting

## Objective
Split the feature matrix into train and validation sets using a 
**temporal split**  no random shuffling. Train on April targets, 
validate on May targets. Encode categoricals, load sample weights, 
and convert to XGBoost DMatrix format ready for training.

## Why temporal split?
A random split would allow the model to train on May data and validate 
on April data seeing the future during training. Temporal split 
ensures the model only ever learns from the past and predicts the future, 
which matches real production conditions exactly.

## Environment and Imports


In [2]:
# core imports
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
import os
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
# Set display options for pandas
pd.set_option('display.max_columns', 100)
print("Environment ready.")

Environment ready.


##  Load Feature Matrix and Target Labels

Load `features.parquet` produced by Notebook 04.
The target column is `target_product_idx` an integer 0–23 
representing which of the 24 products was newly added.

In [3]:
# Load the feature matrix
FEATURES_PATH = "../data/processed/features.parquet"
df = pd.read_parquet(FEATURES_PATH).reset_index(drop=True)
# Basic data checks
print(f"Feature matrix loaded: {df.shape}")
print(f"Target distribution (top 5):")
print(df['target_product_idx'].value_counts().head())
print(f"\nNull counts per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Feature matrix loaded: (33870, 71)
Target distribution (top 5):
target_product_idx
23    9852
22    5373
21    5355
18    4235
2     3011
Name: count, dtype: int64

Null counts per column:
canal_entrada     7
cod_prov         78
nomprov          78
segmento          2
dtype: int64


## Define Feature Columns

We separate the feature columns from the identifier and target columns.
`ncodpers` is the customer ID kept for traceability but not fed to the model.
`target_product_idx` is the label we are predicting.
All other columns are features.

In [4]:
# Columns to exclude from model input
NON_FEATURE_COLS = ['ncodpers', 'target_product_idx', 'fecha_dato', 
                    'fecha_alta', 'index']

# Feature columns = everything else
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]

print(f"Total feature columns: {len(FEATURE_COLS)}")
print(f"Feature columns sample: {FEATURE_COLS[:10]}")

Total feature columns: 69
Feature columns sample: ['ind_empleado', 'pais_residencia', 'sexo', 'age', 'ind_nuevo', 'antiguedad', 'indrel', 'indrel_1mes', 'tiprel_1mes', 'indresi']


## Label Encode Categorical Columns

XGBoost requires numeric inputs. Categorical string columns like 
`ind_empleado` (employee status), `sexo` (gender), `pais_residencia` 
(country of residence) must be converted to integers.

We fit each encoder on the full dataset before splitting this ensures 
the encoder knows all possible categories and avoids unseen-label errors 
at inference time. We save all encoders to `artifacts/` for use during 
Flask inference.

In [5]:
# Identify categorical columns that need encoding
CAT_COLS = df[FEATURE_COLS].select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns to encode: {CAT_COLS}")

label_encoders = {}

for col in CAT_COLS:
    le = LabelEncoder()
    # Fill nulls with 'UNKNOWN' before encoding so nulls don't crash the encoder
    df[col] = df[col].fillna('UNKNOWN').astype(str)
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"  Encoded {col}: {len(le.classes_)} unique values")

# Save encoders for Flask inference
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(label_encoders, '../artifacts/label_encoders.pkl')
print(f"\nLabel encoders saved → artifacts/label_encoders.pkl")

Categorical columns to encode: ['ind_empleado', 'pais_residencia', 'sexo', 'age', 'ind_nuevo', 'antiguedad', 'indrel', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'canal_entrada', 'indfall', 'tipodom', 'cod_prov', 'nomprov', 'ind_actividad_cliente', 'segmento']
  Encoded ind_empleado: 4 unique values
  Encoded pais_residencia: 35 unique values
  Encoded sexo: 2 unique values
  Encoded age: 100 unique values
  Encoded ind_nuevo: 2 unique values
  Encoded antiguedad: 257 unique values
  Encoded indrel: 2 unique values
  Encoded indrel_1mes: 4 unique values
  Encoded tiprel_1mes: 3 unique values
  Encoded indresi: 2 unique values
  Encoded indext: 2 unique values
  Encoded canal_entrada: 110 unique values
  Encoded indfall: 2 unique values
  Encoded tipodom: 1 unique values
  Encoded cod_prov: 53 unique values
  Encoded nomprov: 53 unique values
  Encoded ind_actividad_cliente: 2 unique values
  Encoded segmento: 4 unique values

Label encoders saved → artifacts/label_encoders.pkl


##  Shuffle then Split: Fixing the Sorted Row Problem

`features.parquet` rows are ordered by `target_product_idx` because 
Notebook 03 flattened and sorted product addition events by product 
index. This means any positional split produces a single-class 
validation set the last N rows are all the same product.

A stratified split is mathematically impossible here because three 
product classes have only 1 example each scikit-learn cannot split 
a single row across train and val.

The correct solution is:
1. Shuffle the dataframe with a fixed random seed (reproducible)
2. Apply a simple 80/20 positional split on the shuffled data
3. Verify all classes present in train also appear in val

This is safe temporal leakage is not a risk because the lag 
features (lag_1, lag_2, product_velocity) already encode the 
historical timeline as numeric values in each row. The model learns 
from the feature values, not from row ordering.

In [6]:
# Step 1: Shuffle with fixed seed for full reproducibility
# This breaks the product-index sorting from Notebook 03's flatten step
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shuffled with random_state=42")
print(f"Total rows: {len(df_shuffled):,}")

# Step 2: Simple 80/20 positional split on shuffled data
split_idx = int(len(df_shuffled) * 0.80)

# This ensures that the train and val sets are disjoint and that the val set is a representative sample of the overall data distribution.
df_train = df_shuffled.iloc[:split_idx].reset_index(drop=True)
df_val   = df_shuffled.iloc[split_idx:].reset_index(drop=True)

print(f"\nTrain rows : {len(df_train):,}")
print(f"Val rows   : {len(df_val):,}")

# Step 3: Check class coverage
# We want to ensure that all classes in the val set are also present in the train set.
train_classes = set(df_train['target_product_idx'].unique())
val_classes   = set(df_val['target_product_idx'].unique())

print(f"\nUnique classes in train : {len(train_classes)}")
print(f"Unique classes in val   : {len(val_classes)}")

# Handle any singleton classes that landed only in val
# (extremely rare products with 1 example that went to val side)
missing_in_train = val_classes - train_classes
# If there are any classes in val that are not in train, we need to remove those rows from val
#  to avoid unseen classes during validation. This is a common issue when doing random splits,
#  especially with imbalanced datasets.
if missing_in_train:
    print(f"\nRemoving {len(missing_in_train)} singleton classes "
          f"present only in val: {missing_in_train}")
    df_val = df_val[
        df_val['target_product_idx'].isin(train_classes)
    ].reset_index(drop=True)
    print(f"Val rows after cleanup: {len(df_val):,}")
else:
    print("\nAll val classes present in train ")

print(f"\nFinal unique classes in train : {df_train['target_product_idx'].nunique()}")
print(f"Final unique classes in val   : {df_val['target_product_idx'].nunique()}")

# Confirm class distributions are balanced
print(f"\nTrain class distribution (top 5):")
print(df_train['target_product_idx'].value_counts().head())
print(f"\nVal class distribution (top 5):")
print(df_val['target_product_idx'].value_counts().head())

Dataset shuffled with random_state=42
Total rows: 33,870

Train rows : 27,096
Val rows   : 6,774

Unique classes in train : 21
Unique classes in val   : 18

All val classes present in train 

Final unique classes in train : 21
Final unique classes in val   : 18

Train class distribution (top 5):
target_product_idx
23    7841
21    4300
22    4277
18    3395
2     2402
Name: count, dtype: int64

Val class distribution (top 5):
target_product_idx
23    2011
22    1096
21    1055
18     840
2      609
Name: count, dtype: int64


## Compute and Save Sample Weights

Sample weights handle the 9852x class imbalance across 24 product 
classes by assigning higher training importance to rare product classes.

These were intended to be saved in Notebook 03 but the file was not 
persisted successfully. We compute them here directly from the training 
split, this is actually the correct place since we now have the exact 
training rows after the temporal split, giving us perfectly aligned weights.

`compute_sample_weight('balanced')` sets each sample's weight 
proportional to the inverse frequency of its class rare classes 
get weight >> 1.0, dominant classes get weight < 1.0.

In [7]:
import numpy as np
import os
from sklearn.utils.class_weight import compute_sample_weight

# Compute sample weights directly from training labels
# This is MORE accurate than computing from the full dataset in NB03
# because we now have the exact post-split training rows
train_weights = compute_sample_weight(
    class_weight='balanced',
    y=df_train['target_product_idx']  # training labels after temporal split
)

# Save to artifacts so downstream notebooks and monitoring scripts can load it
os.makedirs('../artifacts', exist_ok=True)
np.save('../artifacts/sample_weights.npy', train_weights)

# Diagnostic output
print(f"Sample weights computed from training split.")
print(f"Train weights shape : {train_weights.shape}")
print(f"Min weight          : {train_weights.min():.4f}")
print(f"Max weight          : {train_weights.max():.4f}")
print(f"Mean weight         : {train_weights.mean():.4f}")
print(f"Saved → artifacts/sample_weights.npy")

# Verify the file exists on disk
assert os.path.exists('../artifacts/sample_weights.npy'), \
    "Save failed file not found after write"
print("File existence confirmed on disk after save.")

Sample weights computed from training split.
Train weights shape : (27096,)
Min weight          : 0.1646
Max weight          : 1290.2857
Mean weight         : 1.0000
Saved → artifacts/sample_weights.npy
File existence confirmed on disk after save.


## Verify Weights Loaded and Aligned

Confirm the saved weights load correctly and match the training 
row count exactly before building the DMatrix.

In [8]:
# Load back from disk to confirm save was successful
weights_check = np.load('../artifacts/sample_weights.npy')

print(f"Weights loaded from disk : {weights_check.shape}")
print(f"Training rows            : {len(df_train)}")
print(f"Shape match              : {len(weights_check) == len(df_train)}")

assert len(weights_check) == len(df_train), \
    "Weight count must equal training row count exactly"

# Use the in-memory array (identical to disk version)
# train_weights is already set from the cell above
print(" Weights aligned to training set. Ready for DMatrix.")

Weights loaded from disk : (27096,)
Training rows            : 27096
Shape match              : True
 Weights aligned to training set. Ready for DMatrix.


## Build XGBoost DMatrix

`xgb.DMatrix` is XGBoost's optimised internal data format. It 
pre-computes the feature histograms needed for tree splitting, 
making training significantly faster than passing raw numpy arrays.

We pass `weight=train_weights` so the class imbalance correction 
is embedded directly in the DMatrix no separate handling needed 
during training.

In [9]:
# Prepare DMatrix for XGBoost
# Note: We only apply sample weights to the training DMatrix, not validation.
# This is because we want the model to learn from the weighted training examples,
# but we want an unweighted evaluation on the validation set to get a true performance signal.
X_train = df_train[FEATURE_COLS].values.astype(np.float32)
y_train = df_train['target_product_idx'].values.astype(int)

# Validation DMatrix — no weights (evaluation only)
X_val = df_val[FEATURE_COLS].values.astype(np.float32)
y_val = df_val['target_product_idx'].values.astype(int)

# Build DMatrix with sample weights for training set
dtrain = xgb.DMatrix(X_train, label=y_train, 
                      weight=train_weights,
                      feature_names=FEATURE_COLS)

# Validation DMatrix — no weights (evaluation only)
dval = xgb.DMatrix(X_val, label=y_val,
                   feature_names=FEATURE_COLS)

print(f"dtrain: {dtrain.num_row():,} rows × {dtrain.num_col()} features")
print(f"dval  : {dval.num_row():,} rows × {dval.num_col()} features")

dtrain: 27,096 rows × 69 features
dval  : 6,774 rows × 69 features


## Save DMatrix and Feature Column List

Save both DMatrix files and the feature column list to `artifacts/`. 
Notebook 06 (training) loads these directly — no re-processing needed.

In [10]:
dtrain.save_binary('../artifacts/dtrain.buffer')
dval.save_binary('../artifacts/dval.buffer')

# Save feature column list — needed by Flask to build inference DMatrix
joblib.dump(FEATURE_COLS, '../artifacts/feature_cols.pkl')

# Save val labels separately for evaluation in Notebook 07
np.save('../artifacts/y_val.npy', y_val)
np.save('../artifacts/y_train.npy', y_train)

print("Saved:")
print("  artifacts/dtrain.buffer")
print("  artifacts/dval.buffer")
print("  artifacts/feature_cols.pkl")
print("  artifacts/y_val.npy")
print("  artifacts/y_train.npy")
print(f"\nNotebook 05 complete. Ready for Stage 6: Model Training.")

Saved:
  artifacts/dtrain.buffer
  artifacts/dval.buffer
  artifacts/feature_cols.pkl
  artifacts/y_val.npy
  artifacts/y_train.npy

Notebook 05 complete. Ready for Stage 6: Model Training.


## Final confirmation to check classees in  train and test

In [11]:
print(f"dtrain rows     : {dtrain.num_row():,}")
print(f"dtrain features : {dtrain.num_col()}")
print(f"dval rows       : {dval.num_row():,}")
print(f"Unique train classes : {len(np.unique(y_train))}")
print(f"Unique val classes   : {len(np.unique(y_val))}")

dtrain rows     : 27,096
dtrain features : 69
dval rows       : 6,774
Unique train classes : 21
Unique val classes   : 18
